In [1]:
import numpy as np
import matplotlib.pyplot as plt
import requests, zipfile, io, os
import pandas as pd

Matplotlib is building the font cache; this may take a moment.
/Users/puddu/Documents/GitHub/AirML/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [5]:

domain = "datasets.techmatrix.it/airml"
token = "DI_xeno_2026"

cities = ["sicilia", "trentino", "venezia", "roma", "puglia",
          "napoli", "firenze", "milano", "bergamo", "bologna"]

for city in cities:
    url = f"https://{domain}/{city}.zip?token={token}"
    resp = requests.get(url, stream=True)
    if resp.ok:
        zip_path = f"./data/{city}.zip"
        if not os.path.exists("./data"):
            os.makedirs("./data")
        with open(zip_path, "wb") as f:
            for chunk in resp.iter_content(8192):
                if chunk:
                    f.write(chunk)
        with zipfile.ZipFile(zip_path, "r") as z:
            z.extractall(path=f"./data")
        os.remove(zip_path)
    else:
        print(f"Failed to download {city}: {resp.status_code}")

In [6]:
dfs = []
for sub in os.listdir("./data"):
    subpath = os.path.join("./data", sub)
    if not os.path.isdir(subpath):
        continue
    csv_path = os.path.join(subpath, "listings.csv")
    if os.path.exists(csv_path):
        dfs.append(pd.read_csv(csv_path))
    else:
        for root, _, files in os.walk(subpath):
            if "listings.csv" in files:
                dfs.append(pd.read_csv(os.path.join(root, "listings.csv")))
                break

if dfs:
    listings = pd.concat(dfs, ignore_index=True)
else:
    listings = pd.DataFrame()

listings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 217222 entries, 0 to 217221
Data columns (total 18 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   id                              217222 non-null  int64  
 1   name                            217222 non-null  object 
 2   host_id                         217222 non-null  int64  
 3   host_name                       217069 non-null  object 
 4   neighbourhood_group             57143 non-null   object 
 5   neighbourhood                   217222 non-null  object 
 6   latitude                        217222 non-null  float64
 7   longitude                       217222 non-null  float64
 8   room_type                       217222 non-null  object 
 9   price                           194273 non-null  float64
 10  minimum_nights                  217222 non-null  int64  
 11  number_of_reviews               217222 non-null  int64  
 12  last_review     